In [36]:
import pandas as pd
import numpy as np
import dask.dataframe as dd
import dask
import glob
import pickle
import numpy as np
from scipy.optimize import brentq
from scipy.stats import norm
from datetime import datetime, time

In [37]:
# Load the CSV
spot_df = dd.read_csv(
    "/home/newberry3/main/_NIFTY_IDX__202507041318.csv",
    usecols=["Date", "Time", "Open", "High", "Low", "Close"]
)

# Create Datetime column using assign (Dask best practice)
spot_df = spot_df.assign(
    Datetime=dd.to_datetime(spot_df["Date"].astype(str) + " " + spot_df["Time"].astype(str))
)

# Drop original Date and Time columns and reorder
spot_df = spot_df[["Datetime", "Open", "High", "Low", "Close"]]

# View first few rows (for debug, does not trigger full compute)
print(spot_df.head())


# Load pickles to Dask DataFrame
options_files = glob.glob("/home/newberry3/Data/NIFTY/NIFTY_*.pkl")
print("Files found:", options_files)
if not options_files:
    raise FileNotFoundError("No .pkl files found in /home/newberry3/Data/NIFTY/")

def load_pickle(file_path):
    with open(file_path, "rb") as f:
        return pickle.load(f)

delayed_dfs = [dask.delayed(load_pickle)(file) for file in options_files]
options_df = dd.from_delayed(delayed_dfs)
options_df.head()

# Save spot_df and options_df to Parquet folders
try:
    spot_df.to_parquet("nifty_spot_data", write_index=False)
except Exception as e:
    print(f"Error saving spot_df: {e}")

try:
    options_df.to_parquet("nifty_options_data", write_index=False)
except Exception as e:
    print(f"Error saving options_df: {e}")

# Reload and preview
try:
    spot_df = dd.read_parquet("nifty_spot_data")
    print("spot_df loaded successfully.")
except Exception as e:
    print(f"Error reading spot_df: {e}")
    spot_df = None

try:
    options_df = dd.read_parquet("nifty_options_data")
    print("options_df loaded successfully.")
except Exception as e:
    print(f"Error reading options_df: {e}")
    options_df = None

if spot_df is not None:
    print("spot_df preview:")
    display(spot_df.head())

if options_df is not None:
    print("options_df preview:")
    display(options_df.head())

options_files = glob.glob("/home/newberry3/Data/NIFTY/NIFTY_*.pkl")
all_cols = set()
dfs = []

# First pass: Find all columns used in any file
for file in options_files:
    df = pickle.load(open(file, "rb"))
    df = df.drop(columns=[col for col in ['OI', 'Volume'] if col in df.columns], errors='ignore')
    all_cols.update(df.columns)

# Make a sorted, consistent list of columns (for Dask, order matters!)
all_cols = sorted(list(all_cols))

# Second pass: Load each DataFrame and align columns (fill missing with NaN)
for file in options_files:
    df = pickle.load(open(file, "rb"))
    df = df.drop(columns=[col for col in ['OI', 'Volume'] if col in df.columns], errors='ignore')
    for col in all_cols:
        if col not in df.columns:
            df[col] = pd.NA
    df = df[all_cols]
    dfs.append(df)

# Combine all into one DataFrame
options_df = pd.concat(dfs, ignore_index=True)

# -- All preprocessing in pandas as above, then:
for col in ['StrikePrice', 'ExpiryDate', 'Ticker', 'Date', 'Time', 'Type']:
    if col in options_df.columns:
        options_df[col] = options_df[col].astype(str)

options_dd = dd.from_pandas(options_df, npartitions=16)
options_dd.to_parquet("nifty_options_data", write_index=False)

# When reading again:
options_dd = dd.read_parquet("nifty_options_data")

print("Columns in combined options_df:", options_df.columns)
print("Earliest date:", options_df['Date'].min(), "Latest date:", options_df['Date'].max())
print("Earliest time:", options_df['Time'].min(), "Latest time:", options_df['Time'].max())
print("Max Datetime:", pd.to_datetime(options_df['Date'] + " " + options_df['Time']).max())

# After reading all files:
print("Earliest and latest Datetime in loaded options_df:")
print(options_df['Date'].min(), options_df['Date'].max())


             Datetime      Open      High       Low     Close
0 2024-05-28 13:57:00  22933.20  22934.35  22926.00  22928.75
1 2024-05-28 13:58:00  22929.05  22933.15  22928.15  22929.60
2 2024-05-28 13:59:00  22929.85  22933.40  22927.50  22929.05
3 2024-05-28 14:00:00  22929.20  22929.55  22921.40  22921.90
4 2024-05-28 14:01:00  22921.50  22931.85  22917.85  22928.20
Files found: ['/home/newberry3/Data/NIFTY/NIFTY_202411.pkl', '/home/newberry3/Data/NIFTY/NIFTY_202412.pkl', '/home/newberry3/Data/NIFTY/NIFTY_202401.pkl', '/home/newberry3/Data/NIFTY/NIFTY_202410.pkl', '/home/newberry3/Data/NIFTY/NIFTY_202502.pkl', '/home/newberry3/Data/NIFTY/NIFTY_202405.pkl', '/home/newberry3/Data/NIFTY/NIFTY_202408.pkl', '/home/newberry3/Data/NIFTY/NIFTY_202404.pkl', '/home/newberry3/Data/NIFTY/NIFTY_202402.pkl', '/home/newberry3/Data/NIFTY/NIFTY_202406.pkl', '/home/newberry3/Data/NIFTY/NIFTY_202501.pkl', '/home/newberry3/Data/NIFTY/NIFTY_202409.pkl', '/home/newberry3/Data/NIFTY/NIFTY_202403.pkl', '/h

,Datetime,Open,High,Low,Close
0,2024-05-28 13:57:00,22933.20,22934.35,22926.00,22928.75
1,2024-05-28 13:58:00,22929.05,22933.15,22928.15,22929.60
2,2024-05-28 13:59:00,22929.85,22933.40,22927.50,22929.05
3,2024-05-28 14:00:00,22929.20,22929.55,22921.40,22921.90
4,2024-05-28 14:01:00,22921.50,22931.85,22917.85,22928.20


options_df preview:


,Close,Date,ExpiryDate,High,Low,Open,StrikePrice,Ticker,Time,Type
0,1713.95,2024-11-04,2024-11-07,1713.95,1713.95,1713.95,22450,2024110409:30NIFTY24110722450CE,09:30,CE
1,1441.25,2024-11-04,2024-11-07,1455.45,1441.25,1455.45,22450,2024110413:58NIFTY24110722450CE,13:58,CE
2,1441.25,2024-11-04,2024-11-07,1441.25,1441.25,1441.25,22450,2024110414:00NIFTY24110722450CE,14:00,CE
3,1.50,2024-11-04,2024-11-07,1.95,1.30,1.95,22450,2024110409:15NIFTY24110722450PE,09:15,PE
4,2.10,2024-11-04,2024-11-07,2.10,1.50,1.50,22450,2024110409:16NIFTY24110722450PE,09:16,PE


Columns in combined options_df: Index(['Close', 'Date', 'ExpiryDate', 'High', 'Low', 'Open', 'StrikePrice',
       'Ticker', 'Time', 'Type'],
      dtype='object')
Earliest date: 2024-01-01 Latest date: 2025-02-28
Earliest time: 09:14 Latest time: 15:31
Max Datetime: 2025-02-28 15:30:00
Earliest and latest Datetime in loaded options_df:
2024-01-01 2025-02-28


In [38]:
import pandas as pd

# --- 1. Load spot minute data (already saved as Parquet in earlier steps) ---
spot_df = pd.read_parquet("nifty_spot_data")

# --- 2. Ensure Datetime is datetime and sort for time-based ops ---
spot_df['Datetime'] = pd.to_datetime(spot_df['Datetime'])
spot_df = spot_df.sort_values('Datetime')

# --- 3. Set Datetime as index for resampling ---
spot_df = spot_df.set_index('Datetime')

# --- 4. Filter to regular NIFTY trading hours (avoid pre/post-market ticks) ---
spot_df = spot_df.between_time('09:15:00', '15:15:00')

# --- 5. Resample to 60-min OHLCV bars. Offset=15min for NIFTY standard (9:15 open) ---
spot_60min = spot_df.resample(
    '60min', 
    origin='start_day', 
    offset='15min', 
    label='left', 
    closed='left'
).agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last'
}).dropna()

# --- 6. Reset index back to columns for easier future ops ---
spot_60min = spot_60min.reset_index()

# --- 7. Calculate 200-period EMA on Close for trend filter ---
spot_60min['EMA_200'] = spot_60min['Close'].ewm(span=200, adjust=False).mean()

# --- 8. Save result for next steps or future notebook runs ---
spot_60min.to_parquet("nifty_spot_data_60min_with_ema.parquet", index=False)

# --- 9. Preview final 60-min OHLC + EMA DataFrame ---
print(spot_60min.tail())


                Datetime      Open      High       Low     Close       EMA_200
6973 2025-06-13 11:15:00  24652.75  24702.10  24639.75  24688.95  24673.173535
6974 2025-06-13 12:15:00  24688.80  24724.60  24674.45  24686.00  24673.301161
6975 2025-06-13 13:15:00  24685.20  24754.35  24680.65  24715.85  24673.724533
6976 2025-06-13 14:15:00  24714.35  24728.45  24650.90  24725.60  24674.240707
6977 2025-06-13 15:15:00  24725.15  24725.35  24709.75  24711.70  24674.613436


In [39]:
# SIGNAL GENERATION

# the logic i am using are:
# the trades are positional, carried forward to next day until exit signal is triggered.
# first check if the price is above the ema 200.
# if it is above we trigger bullish signal
# create a rolling window of 7 candles,
# entry signal : enter when low of candle is less than last 7 bars AND close is less than lowest close of last 7. 
# with bullish entry signal: go long on the open price of the next candle.
# exit signal: when high of candle is greater than last 7 high AND close is greater than highest of last 7 closes.
# bullish reentries are allowed, if new entry signals are generated and then exit all open positions on next exit signal. 

# if the price is below ema 200
# then consider only bearish signals
# entry signal : enter when high of candle is less than last 7 bars AND close is higher than highest close of last 7. 
# with bullish entry signal: go short on the open price of the next candle.
# exit signal: when low of candle is greater than last 7 low AND close is lower than lowest of last 7 closes.
# bearish reentries are allowed, if new entry signals are generated and then exit all open positions on next exit signal.

In [40]:
# Load the 60min candles with EMA (from previous cell)
spot_60min = pd.read_parquet("nifty_spot_data_60min_with_ema.parquet")

# Make sure we're sorted (should be already)
spot_60min = spot_60min.sort_values('Datetime').reset_index(drop=True)

# --- FILTER FOR 2024 ONLY ---
spot_60min = spot_60min[
    (spot_60min['Datetime'] >= pd.Timestamp('2024-01-01')) &
    (spot_60min['Datetime'] < pd.Timestamp('2025-01-01'))
].reset_index(drop=True)


# Pre-calculate rolling indicators
spot_60min['7bar_High'] = spot_60min['High'].shift(1).rolling(window=7).max()
spot_60min['7bar_Low'] = spot_60min['Low'].shift(1).rolling(window=7).min()
spot_60min['7bar_Close_High'] = spot_60min['Close'].shift(1).rolling(window=7).max()
spot_60min['7bar_Close_Low'] = spot_60min['Close'].shift(1).rolling(window=7).min()

# Prepare signal columns
spot_60min['Signal'] = np.nan
spot_60min['Direction'] = np.nan

signals = []

in_position = None # None, "Bullish", or "Bearish"
open_trades = []   # To track multiple entries for reentry logic

for i in range(len(spot_60min)-1):  # always enter/exit on next candle!
    row = spot_60min.iloc[i]
    next_row = spot_60min.iloc[i+1]
    dt = next_row['Datetime']
    ema = row['EMA_200']
    close = row['Close']
    low = row['Low']
    high = row['High']
    
    # --- EMA filter: Only allow one side at a time ---
    if close > ema:
        # BULLISH SIGNALS (can only long, never short if above EMA)
        entry_cond = (low < row['7bar_Low']) and (close < row['7bar_Close_Low'])
        exit_cond = (high > row['7bar_High']) and (close > row['7bar_Close_High'])
        
        if in_position != "Bullish" and entry_cond:
            signals.append({'Datetime': dt, 'Signal': 'Bullish_Entry', 'Direction': 'Bullish'})
            in_position = "Bullish"
            open_trades = ['Bullish_Entry']
        elif in_position == "Bullish" and entry_cond:
            signals.append({'Datetime': dt, 'Signal': 'Bullish_Reentry', 'Direction': 'Bullish'})
            open_trades.append('Bullish_Reentry')
        elif in_position == "Bullish" and exit_cond:
            signals.append({'Datetime': dt, 'Signal': 'Bullish_Exit', 'Direction': 'Bullish'})
            in_position = None
            open_trades = []

    elif close < ema:
        # BEARISH SIGNALS (can only short, never long if below EMA)
        entry_cond = (high > row['7bar_High']) and (close > row['7bar_Close_High'])
        exit_cond = (low < row['7bar_Low']) and (close < row['7bar_Close_Low'])
        
        if in_position != "Bearish" and entry_cond:
            signals.append({'Datetime': dt, 'Signal': 'Bearish_Entry', 'Direction': 'Bearish'})
            in_position = "Bearish"
            open_trades = ['Bearish_Entry']
        elif in_position == "Bearish" and entry_cond:
            signals.append({'Datetime': dt, 'Signal': 'Bearish_Reentry', 'Direction': 'Bearish'})
            open_trades.append('Bearish_Reentry')
        elif in_position == "Bearish" and exit_cond:
            signals.append({'Datetime': dt, 'Signal': 'Bearish_Exit', 'Direction': 'Bearish'})
            in_position = None
            open_trades = []
    # If close == ema, do nothing

# Output all signals as a DataFrame
signals_df = pd.DataFrame(signals)
signals_df = signals_df[['Datetime', 'Signal', 'Direction']]
signals_df.to_parquet("7hl_signals.parquet", index=False)
print(signals_df.head(20))
print(f"Total signals generated: {len(signals_df)}")

              Datetime           Signal Direction
0  2024-01-02 10:15:00    Bullish_Entry   Bullish
1  2024-01-02 11:15:00  Bullish_Reentry   Bullish
2  2024-01-03 11:15:00  Bullish_Reentry   Bullish
3  2024-01-03 15:15:00  Bullish_Reentry   Bullish
4  2024-01-04 11:15:00     Bullish_Exit   Bullish
5  2024-01-08 11:15:00    Bullish_Entry   Bullish
6  2024-01-08 12:15:00  Bullish_Reentry   Bullish
7  2024-01-08 15:15:00  Bullish_Reentry   Bullish
8  2024-01-09 13:15:00     Bullish_Exit   Bullish
9  2024-01-16 13:15:00    Bullish_Entry   Bullish
10 2024-01-17 10:15:00  Bullish_Reentry   Bullish
11 2024-01-17 12:15:00  Bullish_Reentry   Bullish
12 2024-01-17 13:15:00  Bullish_Reentry   Bullish
13 2024-01-17 15:15:00  Bullish_Reentry   Bullish
14 2024-01-19 10:15:00     Bullish_Exit   Bullish
15 2024-01-20 15:15:00    Bullish_Entry   Bullish
16 2024-01-23 11:15:00  Bullish_Reentry   Bullish
17 2024-01-24 15:15:00     Bullish_Exit   Bullish
18 2024-01-30 14:15:00    Bullish_Entry   Bullish


In [41]:
#MAKE TRADELOGS
import pandas as pd

# Load your 60-min candles and signals (if not already in memory)
spot_60min = pd.read_parquet("nifty_spot_data_60min_with_ema.parquet")
signals_df = pd.read_parquet("7hl_signals.parquet")

# Ensure correct types/sorting
spot_60min['Datetime'] = pd.to_datetime(spot_60min['Datetime'])
signals_df['Datetime'] = pd.to_datetime(signals_df['Datetime'])
spot_60min = spot_60min.sort_values('Datetime').reset_index(drop=True)
signals_df = signals_df.sort_values('Datetime').reset_index(drop=True)

# Merge to get prices for signal times
signals_df = pd.merge(
    signals_df, 
    spot_60min[['Datetime', 'Open', 'Close']],
    how='left', 
    on='Datetime'
)

trade_log = []
open_trades = []

for idx, row in signals_df.iterrows():
    signal = row['Signal']
    dt = row['Datetime']
    price = row['Open']  # always use open price of the signal candle for entries/exits
    direction = row['Direction']
    
    # ENTRY/REENTRY: add new open trade
    if "Entry" in signal or "Reentry" in signal:
        open_trades.append({
            'Entry Time': dt,
            'Entry Price': price,
            'Side': direction,
            'Signal Type': signal,
            'Signal Index': idx
        })
    # EXIT: close all open trades on this side
    elif "Exit" in signal:
        to_close = [t for t in open_trades if t['Side'] == direction]
        for trade in to_close:
            trade['Exit Time'] = dt
            trade['Exit Price'] = price
            trade['Exit Signal Index'] = idx
            # PnL: Long = Exit - Entry, Short = Entry - Exit
            if direction == 'Bullish':
                trade['PnL'] = trade['Exit Price'] - trade['Entry Price']
            else:
                trade['PnL'] = trade['Entry Price'] - trade['Exit Price']
            trade_log.append(trade)
        open_trades = [t for t in open_trades if t['Side'] != direction]

# If any positions still open at the end, close at last available price
if open_trades:
    final_price = spot_60min.iloc[-1]['Open']
    final_time = spot_60min.iloc[-1]['Datetime']
    for trade in open_trades:
        trade['Exit Time'] = final_time
        trade['Exit Price'] = final_price
        if trade['Side'] == 'Bullish':
            trade['PnL'] = final_price - trade['Entry Price']
        else:
            trade['PnL'] = trade['Entry Price'] - final_price
        trade['Forced Exit'] = True
        trade_log.append(trade)

trade_log_df = pd.DataFrame(trade_log)

# Select clean columns for output
trade_log_df = trade_log_df[
    ['Entry Time', 'Exit Time', 'Side', 'Entry Price', 'Exit Price', 'Signal Type', 'PnL']
]

# Save for further use
trade_log_df.to_parquet("7hl_tradelog_index.parquet", index=False)
print(trade_log_df.tail(20))
print(f"Total trades: {len(trade_log_df)}")
print(f"Win rate: {(trade_log_df['PnL'] > 0).mean() * 100:.2f}% | Avg PnL: {trade_log_df['PnL'].mean():.2f}")


             Entry Time           Exit Time     Side  Entry Price  Exit Price  \
134 2024-09-30 11:15:00 2024-12-05 12:15:00  Bullish     25930.00    24589.70   
135 2024-09-30 12:15:00 2024-12-05 12:15:00  Bullish     25894.80    24589.70   
136 2024-09-30 13:15:00 2024-12-05 12:15:00  Bullish     25877.15    24589.70   
137 2024-09-30 14:15:00 2024-12-05 12:15:00  Bullish     25827.55    24589.70   
138 2024-09-30 15:15:00 2024-12-05 12:15:00  Bullish     25813.00    24589.70   
139 2024-10-01 11:15:00 2024-12-05 12:15:00  Bullish     25745.45    24589.70   
140 2024-10-03 10:15:00 2024-12-05 12:15:00  Bullish     25529.35    24589.70   
141 2024-12-05 11:15:00 2024-12-05 12:15:00  Bullish     24367.60    24589.70   
142 2024-12-09 10:15:00 2024-12-11 11:15:00  Bullish     24585.95    24653.40   
143 2024-12-10 13:15:00 2024-12-11 11:15:00  Bullish     24551.50    24653.40   
144 2024-12-10 14:15:00 2024-12-11 11:15:00  Bullish     24541.25    24653.40   
145 2024-12-12 11:15:00 2024

In [42]:
# BS/IV functions

# --- Black-Scholes Option Price ---
def black_scholes_price(S, K, T, r, sigma, option_type):
    if sigma <= 0 or T <= 0:
        return 0

    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == "call":
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    elif option_type == "put":
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    else:
        return None
    
# --- Implied Volatility from Option Price ---
def implied_volatility(option_price, S, K, T, r, option_type):
    """
    Uses Brent's method to find implied volatility from the market price.
    """
    try:
        return brentq(
            lambda sigma: black_scholes_price(S, K, T, r, sigma, option_type) - option_price,
            a=0.01,
            b=3.0,
            maxiter=1000,
            xtol=1e-6
        )
    except (ValueError, RuntimeError):
        return None
    
# --- Black-Scholes Greeks ---
def black_scholes_greeks(S, K, T, r, sigma, option_type):
    if sigma <= 0 or T <= 0:
        return None

    # Use synthetic future price
    F = S * np.exp(r * T)

    d1 = (np.log(F / K) + 0.5 * sigma ** 2 * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == "call":
        delta = np.exp(-r * T) * norm.cdf(d1)
        theta = (-F * norm.pdf(d1) * sigma / (2 * np.sqrt(T)) - r * K * np.exp(-r * T) * norm.cdf(d2)) / 365
        rho = K * T * np.exp(-r * T) * norm.cdf(d2) / 100
    else:
        delta = -np.exp(-r * T) * norm.cdf(-d1)
        theta = (-F * norm.pdf(d1) * sigma / (2 * np.sqrt(T)) + r * K * np.exp(-r * T) * norm.cdf(-d2)) / 365
        rho = -K * T * np.exp(-r * T) * norm.cdf(-d2) / 100

    gamma = norm.pdf(d1) / (F * sigma * np.sqrt(T))
    vega = F * norm.pdf(d1) * np.sqrt(T) / 100

    return {
        'Delta': round(delta, 5),
        'Gamma': round(gamma, 5),
        'Vega': round(vega, 5),
        'Theta': round(theta, 5),
        'Rho': round(rho, 5)
    }

# --- Time to Expiry (Fractional) ---
def calculate_time_to_expiry(manual_datetime_str, expiry_date_str):
    now = datetime.strptime(manual_datetime_str, "%Y-%m-%d %H:%M:%S")
    expiry_date = datetime.strptime(expiry_date_str, "%d-%m-%y").date()

    market_open = time(9, 15)
    market_close = time(15, 30)
    today = now.date()
    days_left = (expiry_date - today).days

    if days_left <= 0:
        days_left += 1

    total_trading_minutes = (market_close.hour * 60 + market_close.minute) - (market_open.hour * 60 + market_open.minute)
    current_minutes_since_open = (now.hour * 60 + now.minute) - (market_open.hour * 60 + market_open.minute)

    if current_minutes_since_open < 0:
        T = round(days_left / 365, 6)
    elif current_minutes_since_open >= total_trading_minutes:
        T = round(max(0, (days_left - 1) / 365), 6)
    else:
        fraction_of_day_passed = current_minutes_since_open / total_trading_minutes
        T = round((days_left - fraction_of_day_passed) / 365, 6)

    print(f"\nManual Time Entered: {now}")
    print(f"Expiry Date: {expiry_date}")
    print(f"Days to Expiry (with fraction): {days_left - (fraction_of_day_passed if 0 <= current_minutes_since_open < total_trading_minutes else 0):.6f}")
    print(f"Time to Expiry in Years (T): {T}")

    return T

In [43]:
# from scipy.stats import norm
# from scipy.optimize import brentq
# import numpy as np
# from datetime import datetime, timedelta, time

# # --- Your BS/IV/greeks functions here ---

# # Assume: options_dd is your options Dask DataFrame (all options for 2024)
# # trade_log_df is your trade log with Entry Time, Side, etc.

# selected_strikes = []

# for idx, trade in trade_log_df.iterrows():
#     entry_time = trade['Entry Time']
#     side = trade['Side']
#     spot_price = trade['Entry Price']
#     # 1. Get the nearest expiry >= entry_time
#     expiry_candidates = options_dd[options_dd['Date'] >= entry_time.strftime("%Y-%m-%d")]
#     expiry_dates = expiry_candidates['ExpiryDate'].drop_duplicates().compute().tolist()
#     expiry_dates = sorted([datetime.strptime(d, "%Y-%m-%d") for d in expiry_dates])
#     expiry = min([d for d in expiry_dates if d >= entry_time], default=None)
#     if not expiry:
#         continue  # No valid expiry, skip

#     expiry_str = expiry.strftime("%d-%m-%y")  # or whatever your options file uses

#     # 2. Get all strikes for this expiry and type (call/put)
#     opt_type = 'PE' if side == 'Bullish' else 'CE'
#     options_slice = options_dd[
#         (options_dd['ExpiryDate'] == expiry.strftime("%Y-%m-%d")) &
#         (options_dd['Type'] == opt_type) &
#         (options_dd['Date'] == entry_time.strftime("%Y-%m-%d")) &
#         (options_dd['Time'] == entry_time.strftime("%H:%M:%S"))
#     ].compute()
#     strikes = options_slice['StrikePrice'].astype(float).unique()

#     best_delta_diff = np.inf
#     best_strike = None
#     best_greeks = None
#     best_iv = None

#     # 3. For each strike, calculate IV and delta
#     for strike in strikes:
#         option_row = options_slice[options_slice['StrikePrice'].astype(float) == strike]
#         if option_row.empty:
#             continue
#         option_price = float(option_row['Open'].values[0])  # entry at open
#         T = calculate_time_to_expiry(entry_time.strftime("%Y-%m-%d %H:%M:%S"), expiry_str)
#         option_type_bs = 'put' if opt_type == 'PE' else 'call'
#         iv = implied_volatility(option_price, spot_price, strike, T, 0.066, option_type_bs)
#         if not iv:
#             continue
#         greeks = black_scholes_greeks(spot_price, strike, T, 0.066, iv, option_type_bs)
#         if not greeks:
#             continue
#         target_delta = 0.4 if side == 'Bullish' else -0.4
#         delta_diff = abs(greeks['Delta'] - target_delta)
#         if delta_diff < best_delta_diff:
#             best_delta_diff = delta_diff
#             best_strike = strike
#             best_greeks = greeks
#             best_iv = iv

#     selected_strikes.append({
#         'Entry Time': entry_time,
#         'Side': side,
#         'Strike': best_strike,
#         'Delta': best_greeks['Delta'] if best_greeks else None,
#         'IV': best_iv,
#         'Expiry': expiry_str,
#         # you can add more fields if you want
#     })

# selected_strikes_df = pd.DataFrame(selected_strikes)


In [44]:
import pandas as pd
import numpy as np
from datetime import datetime
from scipy.optimize import brentq
from scipy.stats import norm

# --- Place your Black-Scholes, IV, and Greeks functions here (from your earlier code) ---

# Helper to find nearest expiry >= entry time
def get_nearest_expiry(entry_time, expiry_list):
    expiry_list = sorted([datetime.strptime(x, "%Y-%m-%d") for x in expiry_list])
    for ex in expiry_list:
        if ex >= entry_time:
            return ex.strftime("%Y-%m-%d")
    return None

results = []

for i, trade in trade_log_df.iterrows():
    entry_time = trade['Entry Time']
    side = trade['Side']
    spot = trade['Entry Price']
    # Use just the date part for matching
    entry_date = entry_time.strftime("%Y-%m-%d")
    entry_time_str = entry_time.strftime("%H:%M")

    # 1. Find expiry >= entry_time
    expiry_list = options_df.loc[options_df['Date'] >= entry_date, 'ExpiryDate'].unique()
    if len(expiry_list) == 0:
        continue
    expiry = get_nearest_expiry(entry_time, expiry_list)
    if expiry is None:
        continue

    # 2. Get all options for entry time, expiry, and side
    opt_type = 'PE' if side == 'Bullish' else 'CE'
    available_opts = options_df[
        (options_df['Date'] == entry_date) &
        (options_df['Time'] == entry_time_str) &
        (options_df['ExpiryDate'] == expiry) &
        (options_df['Type'] == opt_type)
    ].copy()
    if available_opts.empty:
        continue

    best_delta_diff = np.inf
    best_row = None

    for _, opt in available_opts.iterrows():
        K = float(opt['StrikePrice'])
        option_price = float(opt['Open'])
        T = calculate_time_to_expiry(f"{entry_date} {entry_time_str}:00", datetime.strptime(expiry, "%Y-%m-%d").strftime("%d-%m-%y"))
        option_type_bs = 'put' if opt_type == 'PE' else 'call'
        try:
            iv = implied_volatility(option_price, spot, K, T, 0.066, option_type_bs)
        except Exception:
            iv = None
        if not iv:
            continue
        greeks = black_scholes_greeks(spot, K, T, 0.066, iv, option_type_bs)
        if not greeks:
            continue
        target_delta = -0.4 if side == 'Bullish' else 0.4
        delta_diff = abs(greeks['Delta'] - target_delta)
        if delta_diff < best_delta_diff:
            best_delta_diff = delta_diff
            best_row = {
                'Entry Time': entry_time,
                'Side': side,
                'Strike': K,
                'Expiry': expiry,
                'IV': iv,
                'Delta': greeks['Delta'],
                'Option Type': opt_type,
                'Option Price': option_price
            }

    if best_row:
        results.append(best_row)

strikes_df = pd.DataFrame(results)
print(strikes_df.head())



Manual Time Entered: 2024-01-02 10:15:00
Expiry Date: 2024-01-04
Days to Expiry (with fraction): 1.840000
Time to Expiry in Years (T): 0.005041

Manual Time Entered: 2024-01-02 10:15:00
Expiry Date: 2024-01-04
Days to Expiry (with fraction): 1.840000
Time to Expiry in Years (T): 0.005041

Manual Time Entered: 2024-01-02 10:15:00
Expiry Date: 2024-01-04
Days to Expiry (with fraction): 1.840000
Time to Expiry in Years (T): 0.005041

Manual Time Entered: 2024-01-02 10:15:00
Expiry Date: 2024-01-04
Days to Expiry (with fraction): 1.840000
Time to Expiry in Years (T): 0.005041

Manual Time Entered: 2024-01-02 10:15:00
Expiry Date: 2024-01-04
Days to Expiry (with fraction): 1.840000
Time to Expiry in Years (T): 0.005041

Manual Time Entered: 2024-01-02 10:15:00
Expiry Date: 2024-01-04
Days to Expiry (with fraction): 1.840000
Time to Expiry in Years (T): 0.005041

Manual Time Entered: 2024-01-02 10:15:00
Expiry Date: 2024-01-04
Days to Expiry (with fraction): 1.840000
Time to Expiry in Years

In [45]:
# import pandas as pd
# import numpy as np
# from datetime import datetime
# from scipy.optimize import brentq
# from scipy.stats import norm

# # --- Your Black-Scholes, IV, and Greeks functions here ---

# # Helper to find nearest expiry >= entry time
# def get_nearest_expiry(entry_time, expiry_list):
#     expiry_list = sorted([datetime.strptime(x, "%Y-%m-%d") for x in expiry_list])
#     for ex in expiry_list:
#         if ex >= entry_time:
#             return ex.strftime("%Y-%m-%d")
#     return None

# results = []
# missing_expiry = []
# missing_valid_option = []
# missing_greeks = []
# iv_failed = []

# for i, trade in trade_log_df.iterrows():
#     entry_time = trade['Entry Time']
#     side = trade['Side']
#     spot = trade['Entry Price']
#     entry_date = entry_time.strftime("%Y-%m-%d")
#     entry_time_str = entry_time.strftime("%H:%M")

#     # 1. Find expiry >= entry_time
#     expiry_list = options_df.loc[options_df['Date'] >= entry_date, 'ExpiryDate'].unique()
#     if len(expiry_list) == 0:
#         missing_expiry.append({'Entry Time': entry_time, 'Side': side, 'Reason': 'No expiry list'})
#         continue
#     expiry = get_nearest_expiry(entry_time, expiry_list)
#     if expiry is None:
#         missing_expiry.append({'Entry Time': entry_time, 'Side': side, 'Reason': 'No expiry >= entry_time'})
#         continue

#     # 2. Find options within ±2 minutes of entry time
#     opt_type = 'PE' if side == 'Bullish' else 'CE'
#     # Find all available times for this date, expiry, type
#     possible_opts = options_df[
#         (options_df['Date'] == entry_date) &
#         (options_df['ExpiryDate'] == expiry) &
#         (options_df['Type'] == opt_type)
#     ].copy()
#     # Find times in possible_opts within ±2 minutes of entry_time_str
#     possible_opts['Time_dt'] = pd.to_datetime(possible_opts['Date'] + " " + possible_opts['Time'], errors='coerce')
#     entry_time_dt = pd.to_datetime(entry_date + " " + entry_time_str)
#     possible_opts['MinuteDiff'] = (possible_opts['Time_dt'] - entry_time_dt).dt.total_seconds().abs() / 60
#     nearest_opts = possible_opts[possible_opts['MinuteDiff'] <= 2]

#     if nearest_opts.empty:
#         missing_valid_option.append({
#             'Entry Time': entry_time,
#             'Side': side,
#             'Expiry': expiry,
#             'OptType': opt_type,
#             'Entry TimeStr': entry_time_str,
#             'Available Times': sorted(possible_opts['Time'].unique())
#         })
#         continue

#     best_delta_diff = np.inf
#     best_row = None

#     for _, opt in nearest_opts.iterrows():
#         K = float(opt['StrikePrice'])
#         option_price = float(opt['Open'])
#         match_time = opt['Time']
#         # Use the actual matched time for T calculation
#         time_full_str = f"{entry_date} {match_time}:00"
#         T = calculate_time_to_expiry(time_full_str, datetime.strptime(expiry, "%Y-%m-%d").strftime("%d-%m-%y"))
#         option_type_bs = 'put' if opt_type == 'PE' else 'call'
#         try:
#             iv = implied_volatility(option_price, spot, K, T, 0.066, option_type_bs)
#         except Exception:
#             iv = None
#         if not iv:
#             iv_failed.append({
#                 'Entry Time': entry_time,
#                 'Side': side,
#                 'Expiry': expiry,
#                 'Strike': K,
#                 'Option Price': option_price,
#                 'Spot': spot,
#                 'T': T,
#                 'Matched Option Time': match_time
#             })
#             continue
#         greeks = black_scholes_greeks(spot, K, T, 0.066, iv, option_type_bs)
#         if not greeks:
#             missing_greeks.append({
#                 'Entry Time': entry_time,
#                 'Side': side,
#                 'Expiry': expiry,
#                 'Strike': K,
#                 'Option Price': option_price,
#                 'Spot': spot,
#                 'T': T,
#                 'Matched Option Time': match_time,
#                 'IV': iv
#             })
#             continue
#         target_delta = -0.4 if side == 'Bullish' else 0.4
#         delta_diff = abs(greeks['Delta'] - target_delta)
#         if delta_diff < best_delta_diff:
#             best_delta_diff = delta_diff
#             best_row = {
#                 'Entry Time': entry_time,
#                 'Side': side,
#                 'Strike': K,
#                 'Expiry': expiry,
#                 'IV': iv,
#                 'Delta': greeks['Delta'],
#                 'Option Type': opt_type,
#                 'Option Price': option_price,
#                 'Matched Option Time': match_time
#             }

#     if best_row:
#         results.append(best_row)

# # Save or use your strikes
# strikes_df = pd.DataFrame(results)

# print(f"Total trades in log: {len(trade_log_df)}")
# print(f"Trades with selected option: {len(strikes_df)}")
# print(f"Missing expiry: {len(missing_expiry)}")
# print(f"Missing options at entry time (within ±2 min): {len(missing_valid_option)}")
# print(f"IV calculation failed: {len(iv_failed)}")
# print(f"No greeks: {len(missing_greeks)}")

# if missing_expiry:
#     print("\n--- Missing Expiry Details ---")
#     for e in missing_expiry:
#         print(e)

# if missing_valid_option:
#     print("\n--- Missing Option Data (No match within ±2 min) ---")
#     for o in missing_valid_option:
#         print(o)

# if iv_failed:
#     print("\n--- IV Calculation Failed ---")
#     for ivf in iv_failed:
#         print(ivf)

# if missing_greeks:
#     print("\n--- No Greeks Calculated ---")
#     for mg in missing_greeks:
#         print(mg)


In [46]:
#FIX THURSDAY ISSUE

for m in missing_valid_option:
    print(m['Entry Time'], m['Entry Time'].weekday())


2024-01-20 15:15:00 5
2024-02-08 11:15:00 3
2024-02-08 15:15:00 3
2024-02-22 10:15:00 3
2024-02-22 11:15:00 3
2024-02-29 10:15:00 3
2024-04-18 11:15:00 3
2024-04-18 12:15:00 3
2024-05-16 10:15:00 3
2024-05-02 09:15:00 3
2024-05-30 10:15:00 3
2024-05-30 14:15:00 3
2024-07-25 10:15:00 3
2024-09-05 15:15:00 3
2024-10-03 10:15:00 3
2024-12-05 11:15:00 3
2024-12-12 11:15:00 3
2024-12-12 12:15:00 3
2024-12-12 15:15:00 3


In [47]:
strikes_df

,Entry Time,Side,Strike,Expiry,IV,Delta,Option Type,Option Price
0,2024-01-02 10:15:00,Bullish,21600.0,2024-01-04,0.146279,-0.37879,PE,59.65
1,2024-01-02 11:15:00,Bullish,21550.0,2024-01-04,0.156179,-0.42086,PE,70.70
2,2024-01-03 11:15:00,Bullish,21550.0,2024-01-04,0.179668,-0.44938,PE,56.90
3,2024-01-03 15:15:00,Bullish,21500.0,2024-01-04,0.726640,-0.45959,PE,57.70
4,2024-01-08 11:15:00,Bullish,21550.0,2024-01-11,0.145062,-0.37915,PE,71.45
...,...,...,...,...,...,...,...,...
130,2024-12-02 13:15:00,Bearish,24300.0,2024-12-05,0.218182,0.38521,CE,113.35
131,2024-12-27 10:15:00,Bearish,24050.0,2025-01-02,0.135110,0.40717,CE,118.60
132,2024-12-30 12:15:00,Bearish,24000.0,2025-01-02,0.177902,0.38309,CE,93.80
133,2024-12-17 10:15:00,Bullish,24450.0,2024-12-19,0.228516,-0.41896,PE,122.55


In [ ]:
# #MATCHING STRIKES FROM OPTION DATA TO MATCH ENTRY AND EXIT PREMIUMS AND PNL
# final_trades = []

# for i, row in strikes_df.iterrows():
#     entry_time = row['Entry Time']
#     exit_time = trade_log_df.loc[trade_log_df['Entry Time'] == entry_time, 'Exit Time']
#     if exit_time.empty:
#         continue
#     exit_time = exit_time.values[0]
#     expiry = row['Expiry']
#     strike = row['Strike']
#     option_type = row['Option Type']
#     entry_option_price = row['Option Price']
    
#     # Find option at exit time, nearest within ±2min
#     exit_time = pd.to_datetime(exit_time) 
#     exit_date = exit_time.strftime("%Y-%m-%d")
#     exit_time_str = exit_time.strftime("%H:%M")
#     # Get all available times on that exit date for that strike/expiry/type
#     mask = (
#         (options_df['Date'] == exit_date) &
#         (options_df['ExpiryDate'] == expiry) &
#         (options_df['StrikePrice'].astype(float) == strike) &
#         (options_df['Type'] == option_type)
#     )
#     candidate_times = pd.to_datetime(exit_date + " " + options_df.loc[mask, 'Time'])
#     # Find nearest time
#     if not candidate_times.empty:
#         time_diffs = abs(candidate_times - exit_time)
#         min_idx = time_diffs.idxmin()
#         exit_option_price = options_df.loc[min_idx, 'Open']  # Or 'Close'
#     else:
#         exit_option_price = np.nan

#     # Calculate PnL
#     if row['Side'] == 'Bullish':
#         pnl = exit_option_price - entry_option_price if pd.notnull(exit_option_price) else np.nan
#     else:
#         pnl = entry_option_price - exit_option_price if pd.notnull(exit_option_price) else np.nan

#     final_trades.append({
#         'Entry Time': entry_time,
#         'Exit Time': exit_time,
#         'Side': row['Side'],
#         'Strike': strike,
#         'Expiry': expiry,
#         'Option Type': option_type,
#         'Entry Option Price': entry_option_price,
#         'Exit Option Price': exit_option_price,
#         'PnL': pnl
#     })

# final_trades_df = pd.DataFrame(final_trades)
# final_trades_df.to_parquet("final_option_trades.parquet", index=False)
# print(final_trades_df.head())


KeyboardInterrupt: 

In [ ]:
import pandas as pd

# Check for nulls and types
print("Nulls per column:\n", final_trades_df.isnull().sum())
print("\nDtypes:\n", final_trades_df.dtypes)

# Show sample problematic rows (where PnL or Option Price is NaN)
print("\nRows with missing Entry Option Price:")
print(final_trades_df[final_trades_df['Entry Option Price'].isnull()])

print("\nRows with missing Exit Option Price:")
print(final_trades_df[final_trades_df['Exit Option Price'].isnull()])

print("\nRows with missing PnL:")
print(final_trades_df[final_trades_df['PnL'].isnull()])


Nulls per column:
 Entry Time             0
Exit Time              0
Side                   0
Strike                 0
Expiry                 0
Option Type            0
Entry Option Price     0
Exit Option Price     57
PnL                   57
dtype: int64

Dtypes:
 Entry Time            datetime64[ns]
Exit Time             datetime64[ns]
Side                          object
Strike                       float64
Expiry                        object
Option Type                   object
Entry Option Price           float64
Exit Option Price            float64
PnL                          float64
dtype: object

Rows with missing Entry Option Price:
Empty DataFrame
Columns: [Entry Time, Exit Time, Side, Strike, Expiry, Option Type, Entry Option Price, Exit Option Price, PnL]
Index: []

Rows with missing Exit Option Price:
             Entry Time           Exit Time     Side   Strike      Expiry  \
7   2024-01-16 13:15:00 2024-01-19 10:15:00  Bullish  21950.0  2024-01-18   
8   2024-01-17 10

In [54]:
import pandas as pd
import numpy as np

# Convert for quick access
options_df['StrikePrice'] = options_df['StrikePrice'].astype(float)
options_df['DateTime'] = pd.to_datetime(options_df['Date'] + ' ' + options_df['Time'])
options_df['ExpiryDate'] = pd.to_datetime(options_df['ExpiryDate'])

strikes_df['Entry Time'] = pd.to_datetime(strikes_df['Entry Time'])
strikes_df['Expiry'] = pd.to_datetime(strikes_df['Expiry'])
trade_log_df['Entry Time'] = pd.to_datetime(trade_log_df['Entry Time'])
trade_log_df['Exit Time'] = pd.to_datetime(trade_log_df['Exit Time'])

# Merge exit times into strikes_df
merged_df = strikes_df.merge(trade_log_df[['Entry Time', 'Exit Time']], on='Entry Time', how='left')

def get_exit_price(row):
    strike = row['Strike']
    option_type = row['Option Type']
    expiry = row['Expiry']
    exit_time = row['Exit Time']
    
    # CASE 1: Exit after expiry (or on a day with no data after expiry)
    if exit_time > expiry:
        expiry_str = expiry.strftime("%Y-%m-%d")
        mask = (
            (options_df['ExpiryDate'] == expiry) &
            (options_df['StrikePrice'] == strike) &
            (options_df['Type'] == option_type) &
            (options_df['Date'] == expiry_str)
        )
        expiry_opts = options_df[mask]
        if not expiry_opts.empty:
            last_tick = expiry_opts.sort_values('Time').iloc[-1]
            return last_tick['Close']
        else:
            return np.nan
    # CASE 2: Exit on or before expiry
    else:
        exit_date_str = exit_time.strftime("%Y-%m-%d")
        mask = (
            (options_df['Date'] == exit_date_str) &
            (options_df['ExpiryDate'] == expiry) &
            (options_df['StrikePrice'] == strike) &
            (options_df['Type'] == option_type)
        )
        opts = options_df[mask]
        if opts.empty:
            return np.nan
        opts = opts.copy()
        opts['TimeDT'] = pd.to_datetime(exit_date_str + " " + opts['Time'])
        time_diffs = abs(opts['TimeDT'] - exit_time)
        if time_diffs.empty:
            return np.nan
        # Reset index to align .iloc and .loc with time_diffs
        opts = opts.reset_index(drop=True)
        time_diffs = time_diffs.reset_index(drop=True)
        min_idx = time_diffs.idxmin()
        if time_diffs[min_idx] <= pd.Timedelta(minutes=2):
            return opts.loc[min_idx, 'Open']
        else:
            return np.nan


# Vectorized application
merged_df['Exit Option Price'] = merged_df.apply(get_exit_price, axis=1)

# Calculate PnL
merged_df['PnL'] = np.where(
    merged_df['Side'] == 'Bullish',
    merged_df['Exit Option Price'] - merged_df['Option Price'],
    merged_df['Option Price'] - merged_df['Exit Option Price']
)

# Export
merged_df.to_parquet("final_option_trades_vectorized.parquet", index=False)
print(merged_df.head())
print(merged_df[merged_df['Exit Option Price'].isna()])  # Show trades missing exit price


           Entry Time     Side   Strike     Expiry        IV     Delta  \
0 2024-01-02 10:15:00  Bullish  21600.0 2024-01-04  0.146278 -0.378795   
1 2024-01-02 11:15:00  Bullish  21550.0 2024-01-04  0.156183 -0.420862   
2 2024-01-03 11:15:00  Bullish  21550.0 2024-01-04  0.179667 -0.449376   
3 2024-01-03 15:15:00  Bullish  21500.0 2024-01-04  0.727998 -0.459590   
4 2024-01-08 11:15:00  Bullish  21550.0 2024-01-11  0.145058 -0.379152   

  Option Type  Option Price           Exit Time  Exit Option Price    PnL  
0          PE         59.65 2024-01-04 11:15:00               0.05 -59.60  
1          PE         70.70 2024-01-04 11:15:00               0.05 -70.65  
2          PE         56.90 2024-01-04 11:15:00               0.05 -56.85  
3          PE         57.70 2024-01-04 11:15:00               0.05 -57.65  
4          PE         71.45 2024-01-09 13:15:00              27.70 -43.75  
Empty DataFrame
Columns: [Entry Time, Side, Strike, Expiry, IV, Delta, Option Type, Option Price, E

In [55]:
print("Trade count:", len(merged_df))
print("Winning trades:", (merged_df['PnL'] > 0).sum())
print("Losing trades:", (merged_df['PnL'] < 0).sum())
print(f"Total PnL: {merged_df['PnL'].sum():.2f}")



Trade count: 140
Winning trades: 70
Losing trades: 70
Total PnL: 7061.65


In [51]:
merged_df.to_excel("my_merged_df.xlsx", index=False)

In [52]:
import dask.dataframe as dd
import pandas as pd
import numpy as np
from datetime import datetime, time
from scipy.optimize import brentq
from scipy.stats import norm

# --- Black-Scholes, IV, Greeks from your code ---

def black_scholes_price(S, K, T, r, sigma, option_type):
    if sigma <= 0 or T <= 0:
        return 0
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == "call":
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    elif option_type == "put":
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    else:
        return None

def implied_volatility(option_price, S, K, T, r, option_type):
    try:
        return brentq(
            lambda sigma: black_scholes_price(S, K, T, r, sigma, option_type) - option_price,
            a=0.01,
            b=3.0,
            maxiter=1000,
            xtol=1e-6
        )
    except (ValueError, RuntimeError):
        return None

def black_scholes_greeks(S, K, T, r, sigma, option_type):
    if sigma <= 0 or T <= 0:
        return None
    F = S * np.exp(r * T)
    d1 = (np.log(F / K) + 0.5 * sigma ** 2 * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == "call":
        delta = np.exp(-r * T) * norm.cdf(d1)
    else:
        delta = -np.exp(-r * T) * norm.cdf(-d1)
    return {'Delta': delta}

def calculate_time_to_expiry(entry_time, expiry_str):
    now = pd.to_datetime(entry_time)
    expiry_date = datetime.strptime(expiry_str, "%Y-%m-%d")
    market_open = time(9, 15)
    market_close = time(15, 30)
    days_left = (expiry_date.date() - now.date()).days
    total_trading_minutes = (market_close.hour * 60 + market_close.minute) - (market_open.hour * 60 + market_open.minute)
    current_minutes_since_open = (now.hour * 60 + now.minute) - (market_open.hour * 60 + market_open.minute)
    if current_minutes_since_open < 0:
        T = days_left / 365
    elif current_minutes_since_open >= total_trading_minutes:
        T = max(0, (days_left - 1) / 365)
    else:
        fraction_of_day_passed = current_minutes_since_open / total_trading_minutes
        T = (days_left - fraction_of_day_passed) / 365
    return max(T, 1e-6)  # avoid T=0

# ---- MAIN DASK PIPELINE ----

def find_best_strike(row, options_dd, r=0.066):
    entry_time = row['Entry Time']
    entry_date = pd.to_datetime(entry_time).strftime("%Y-%m-%d")
    entry_time_str = pd.to_datetime(entry_time).strftime("%H:%M")
    spot = row['Entry Price']
    side = row['Side']
    opt_type = 'PE' if side == 'Bullish' else 'CE'
    expiry_list = options_dd.loc[options_dd['Date'] >= entry_date, 'ExpiryDate'].drop_duplicates().compute().tolist()
    expiry_list = sorted([x for x in expiry_list if x >= entry_date])
    if not expiry_list:
        return None
    expiry = expiry_list[0]  # nearest expiry
    # Get all strikes for this expiry, entry date, time, and type
    options = options_dd[
        (options_dd['Date'] == entry_date) &
        (options_dd['Time'] == entry_time_str) &
        (options_dd['ExpiryDate'] == expiry) &
        (options_dd['Type'] == opt_type)
    ].compute()
    if options.empty:
        return None
    best_row = None
    best_delta_diff = np.inf
    target_delta = -0.4 if side == 'Bullish' else 0.4
    for _, opt in options.iterrows():
        K = float(opt['StrikePrice'])
        option_price = float(opt['Open'])
        T = calculate_time_to_expiry(f"{entry_date} {entry_time_str}:00", expiry)
        option_type_bs = 'put' if opt_type == 'PE' else 'call'
        iv = implied_volatility(option_price, spot, K, T, r, option_type_bs)
        if not iv:
            continue
        greeks = black_scholes_greeks(spot, K, T, r, iv, option_type_bs)
        if not greeks:
            continue
        delta_diff = abs(greeks['Delta'] - target_delta)
        if delta_diff < best_delta_diff:
            best_delta_diff = delta_diff
            best_row = {
                'Entry Time': entry_time,
                'Side': side,
                'Strike': K,
                'Expiry': expiry,
                'IV': iv,
                'Delta': greeks['Delta'],
                'Option Type': opt_type,
                'Option Price': option_price
            }
    return best_row

# Convert trade_log_df to dask for efficient mapping
trade_dd = dd.from_pandas(trade_log_df, npartitions=4)

# Map function to every row
results = trade_dd.map_partitions(
    lambda df: df.apply(find_best_strike, axis=1, options_dd=options_dd), meta=('result', 'object')
).compute()

# Remove None results and build strikes_df
strikes_list = [res for res in results if res is not None]
strikes_df = pd.DataFrame(strikes_list)
print(strikes_df.head())


           Entry Time     Side   Strike      Expiry        IV     Delta  \
0 2024-01-02 10:15:00  Bullish  21600.0  2024-01-04  0.146278 -0.378795   
1 2024-01-02 11:15:00  Bullish  21550.0  2024-01-04  0.156183 -0.420862   
2 2024-01-03 11:15:00  Bullish  21550.0  2024-01-04  0.179667 -0.449376   
3 2024-01-03 15:15:00  Bullish  21500.0  2024-01-04  0.727998 -0.459590   
4 2024-01-08 11:15:00  Bullish  21550.0  2024-01-11  0.145058 -0.379152   

  Option Type  Option Price  
0          PE         59.65  
1          PE         70.70  
2          PE         56.90  
3          PE         57.70  
4          PE         71.45  


In [53]:
strikes_df

,Entry Time,Side,Strike,Expiry,IV,Delta,Option Type,Option Price
0,2024-01-02 10:15:00,Bullish,21600.0,2024-01-04,0.146278,-0.378795,PE,59.65
1,2024-01-02 11:15:00,Bullish,21550.0,2024-01-04,0.156183,-0.420862,PE,70.70
2,2024-01-03 11:15:00,Bullish,21550.0,2024-01-04,0.179667,-0.449376,PE,56.90
3,2024-01-03 15:15:00,Bullish,21500.0,2024-01-04,0.727998,-0.459590,PE,57.70
4,2024-01-08 11:15:00,Bullish,21550.0,2024-01-11,0.145058,-0.379152,PE,71.45
...,...,...,...,...,...,...,...,...
135,2024-12-02 13:15:00,Bearish,24300.0,2024-12-05,0.218187,0.385207,CE,113.35
136,2024-12-27 10:15:00,Bearish,24050.0,2025-01-02,0.135110,0.407169,CE,118.60
137,2024-12-30 12:15:00,Bearish,24000.0,2025-01-02,0.177900,0.383091,CE,93.80
138,2024-12-17 10:15:00,Bullish,24450.0,2024-12-19,0.228514,-0.418960,PE,122.55
